## 0. Setup Colab (RODE PRIMEIRO)
Remonta o Drive, instala libs e confirma que os arquivos da avaliação existem.
Rode logo após o treino, no mesmo runtime, para reaproveitar o cache dos modelos.

In [8]:
# ===== SETUP COLAB - AVALIAÇÃO =====
from google.colab import drive
drive.mount("/content/drive")
import os
WORK = "/content/drive/MyDrive/Q4_destilacao"
os.chdir(WORK)
print("Pasta:", os.getcwd())

# libs (mesmas versões do treino + rouge para métricas)
!pip install -q "transformers>=4.45,<4.48" "peft>=0.13,<0.15" datasets accelerate rouge-score bitsandbytes

import torch
print("GPU:", torch.cuda.get_device_name(0) if torch.cuda.is_available() else "SEM GPU")

# confere arquivos necessários
for arq in ["benchmark_q4.json", "split_teste.json", "student_cot_adapter"]:
    print(arq, "OK" if os.path.exists(arq) else "FALTANDO!")

Drive already mounted at /content/drive; to attempt to forcibly remount, call drive.mount("/content/drive", force_remount=True).
Pasta: /content/drive/MyDrive/Q4_destilacao
GPU: Tesla T4
benchmark_q4.json OK
split_teste.json OK
student_cot_adapter OK


# Questão 4 — Avaliação da Destilação (antes vs. depois)
### Tópicos em IA (2026.1) — DC/UFPI
Grupo: Émery Moriconi, Eryck Torres, Felipe Lages, Raffael Fernandes

Este notebook responde à pergunta central da questão: **houve transferência de
conhecimento do teacher para o student?**

Avalia três variantes sobre o **mesmo** benchmark e o **mesmo** conjunto de teste:

| # | Variante | Papel |
|---|----------|-------|
| 1 | Qwen2.5-7B-Instruct (teacher) | teto de referência |
| 2 | Qwen2.5-0.5B-Instruct (student base, **sem** destilação) | baseline — o "antes" |
| 3 | Student 0.5B + adapter CoT (destilado) | o "depois" |

Métricas: **perplexidade** (teste), **acurácia de resposta** no benchmark de 100
(ROUGE-L + termos-chave + LLM-as-judge opcional) e **taxa de português correto**.

## 1. Setup

In [9]:
# !pip install -q transformers peft datasets accelerate rouge-score sacremoses

import os, json, math, re, time
import numpy as np
import torch

SEED = 42
DEVICE = "cuda" if torch.cuda.is_available() else "cpu"

STUDENT_ID  = "Qwen/Qwen2.5-0.5B-Instruct"
TEACHER_ID  = "Qwen/Qwen2.5-7B-Instruct"   # se não couber, ver Seção 7 (Ollama)
ADAPTER_DIR = "student_cot_adapter"
BENCHMARK_PATH = "benchmark_q4.json"        # 100 perguntas + referências
TESTE_PATH     = "split_teste.json"         # gerado no notebook de treino

print("Device:", DEVICE)

Device: cuda


## 2. Benchmark de 100 perguntas
Formato esperado de cada item: `{id, pergunta, resposta_referencia, categoria}`.

> **Atenção a data leakage:** estas 100 perguntas devem vir de chunks que NÃO
> entraram no treino. A célula abaixo cruza o benchmark com o split de treino e
> alerta se houver interseção de instruções.

In [10]:
with open(BENCHMARK_PATH, "r", encoding="utf-8") as f:
    benchmark = json.load(f)
print(f"Benchmark: {len(benchmark)} perguntas")

# Checagem de leakage (se o split de treino estiver disponível)
def norm(s): return re.sub(r"\s+", " ", s.strip().lower())
if os.path.exists("dataset_cot_limpo.json"):
    with open("dataset_cot_limpo.json", encoding="utf-8") as f:
        treino_instr = {norm(p["instruction"]) for p in json.load(f)}
    bench_instr = {norm(b["pergunta"]) for b in benchmark}
    inter = treino_instr & bench_instr
    print(f"Interseção benchmark↔treino: {len(inter)}"
          + ("  ⚠️ REVISAR" if inter else "  ✓ sem leakage óbvio"))

Benchmark: 100 perguntas
Interseção benchmark↔treino: 0  ✓ sem leakage óbvio


## 3. Funções de carregamento dos modelos

In [11]:
from transformers import AutoModelForCausalLM, AutoTokenizer, BitsAndBytesConfig
from peft import PeftModel

def carregar(model_id, adapter=None, quantizar=False):
    tok = AutoTokenizer.from_pretrained(model_id, trust_remote_code=True)
    if tok.pad_token is None:
        tok.pad_token = tok.eos_token
    kwargs = dict(trust_remote_code=True, device_map="auto")
    if quantizar:
        # teacher 7B: 4-bit para caber com folga na T4 (16GB)
        kwargs["quantization_config"] = BitsAndBytesConfig(
            load_in_4bit=True,
            bnb_4bit_compute_dtype=torch.float16,
            bnb_4bit_quant_type="nf4",
            bnb_4bit_use_double_quant=True,
        )
    else:
        kwargs["torch_dtype"] = torch.float16
    mdl = AutoModelForCausalLM.from_pretrained(model_id, **kwargs)
    if adapter:
        mdl = PeftModel.from_pretrained(mdl, adapter)
    mdl.eval()
    return mdl, tok

def liberar(mdl):
    del mdl
    import gc; gc.collect()
    if torch.cuda.is_available():
        torch.cuda.empty_cache()

## 4. Métrica 1 — Perplexidade no conjunto de teste
PPL mede quão bem o modelo prevê o alvo (reasoning + answer). É a métrica
central da Q2/Q3, então mantém comparabilidade. Calculada sobre o **mesmo**
`split_teste.json` para todas as variantes.

In [12]:
def montar_texto_alvo(par):
    msgs = [{"role":"user","content": par["instruction"]}]
    return msgs, f"{par['reasoning'].strip()}\n\nResposta: {par['answer'].strip()}"

@torch.no_grad()
def perplexidade(mdl, tok, pares, max_len=512):
    total_nll, total_tok = 0.0, 0
    for par in pares:
        msgs, alvo = montar_texto_alvo(par)
        prompt_ids = tok.apply_chat_template(msgs, tokenize=True,
                                             add_generation_prompt=True)
        alvo_ids = tok(alvo, add_special_tokens=False)["input_ids"] + [tok.eos_token_id]
        input_ids = (prompt_ids + alvo_ids)[:max_len]
        labels = ([-100]*len(prompt_ids) + alvo_ids)[:max_len]
        ii = torch.tensor([input_ids]).to(mdl.device)
        ll = torch.tensor([labels]).to(mdl.device)
        out = mdl(input_ids=ii, labels=ll)
        n_alvo = (ll != -100).sum().item()
        total_nll += out.loss.item() * n_alvo
        total_tok += n_alvo
    return math.exp(total_nll / total_tok)

with open(TESTE_PATH, encoding="utf-8") as f:
    teste = json.load(f)
print(f"Conjunto de teste: {len(teste)} pares")

Conjunto de teste: 55 pares


## 5. Métrica 2 — Acurácia de resposta (benchmark)
Combina três sinais, porque nenhum sozinho é confiável para texto gerado:
- **ROUGE-L** entre resposta gerada e referência (sobreposição estrutural).
- **Cobertura de termos-chave** da referência (recall lexical de conteúdo).
- **Português correto** (heurística — o baseline 0.5B tende a misturar idioma).

O LLM-as-judge (opcional) fica na Seção 8.

In [13]:
from rouge_score import rouge_scorer
scorer = rouge_scorer.RougeScorer(["rougeL"], use_stemmer=False)

PT_HINTS = (" de "," que "," para "," com "," uma "," os "," as "," da "," do ",
            " em "," é "," na "," no "," são "," ao ")
def pt_ok(t):
    t = f" {t.lower()} "
    return sum(1 for h in PT_HINTS if h in t) >= 2

def termos_chave(ref, k=8):
    palavras = re.findall(r"[a-zà-ú]{4,}", ref.lower())
    vistos, out = set(), []
    for p in palavras:
        if p not in vistos:
            vistos.add(p); out.append(p)
    return out[:k]

def cobertura(resp, ref):
    chaves = termos_chave(ref)
    if not chaves: return 0.0
    rl = resp.lower()
    return sum(1 for c in chaves if c in rl) / len(chaves)

@torch.no_grad()
def gerar_resposta(mdl, tok, pergunta, max_new=256):
    msgs = [{"role":"user","content": pergunta}]
    ids = tok.apply_chat_template(msgs, tokenize=True, add_generation_prompt=True,
                                  return_tensors="pt").to(mdl.device)
    out = mdl.generate(ids, max_new_tokens=max_new, do_sample=False,
                       repetition_penalty=1.15, pad_token_id=tok.pad_token_id)
    return tok.decode(out[0][ids.shape[1]:], skip_special_tokens=True)

def avaliar_benchmark(mdl, tok, bench):
    linhas = []
    for b in bench:
        resp = gerar_resposta(mdl, tok, b["pergunta"])
        ref = b["resposta_referencia"]
        linhas.append({
            "id": b.get("id"),
            "categoria": b.get("categoria",""),
            "rougeL": scorer.score(ref, resp)["rougeL"].fmeasure,
            "cobertura": cobertura(resp, ref),
            "pt_ok": int(pt_ok(resp)),
            "resposta": resp,
        })
    return linhas

def resumir(linhas):
    return {
        "rougeL_medio": float(np.mean([l["rougeL"] for l in linhas])),
        "cobertura_media": float(np.mean([l["cobertura"] for l in linhas])),
        "pct_portugues": float(np.mean([l["pt_ok"] for l in linhas])),
    }

## 6. Avaliar as três variantes
Carrega uma de cada vez e libera a memória antes da próxima (importante em T4).
O teacher é avaliado por último (maior; ver alternativa via Ollama na Seção 7).

In [14]:
resultados = {}

# --- Variante 2: student base (o "antes") ---
print("== Student BASE (sem destilação) ==")
mdl, tok = carregar(STUDENT_ID)
ppl_base = perplexidade(mdl, tok, teste)
bench_base = avaliar_benchmark(mdl, tok, benchmark)
resultados["student_base"] = {"ppl": ppl_base, **resumir(bench_base)}
liberar(mdl)
print(resultados["student_base"])

# --- Variante 3: student destilado (o "depois") ---
print("\n== Student DESTILADO (CoT) ==")
mdl, tok = carregar(STUDENT_ID, adapter=ADAPTER_DIR)
ppl_dest = perplexidade(mdl, tok, teste)
bench_dest = avaliar_benchmark(mdl, tok, benchmark)
resultados["student_destilado"] = {"ppl": ppl_dest, **resumir(bench_dest)}
liberar(mdl)
print(resultados["student_destilado"])

== Student BASE (sem destilação) ==
{'ppl': 6.5240569896058265, 'rougeL_medio': 0.17537966114623818, 'cobertura_media': 0.55875, 'pct_portugues': 1.0}

== Student DESTILADO (CoT) ==
{'ppl': 4.524251921012057, 'rougeL_medio': 0.2260487609949605, 'cobertura_media': 0.40875, 'pct_portugues': 1.0}


### 6.1 Teacher (teto de referência)
Se o 7B couber na GPU, rode a célula abaixo. Caso contrário, use a Seção 7 (Ollama).

In [15]:
# TEACHER 7B (teto de referência) - quantizado em 4-bit para caber na T4
print("== TEACHER 7B (4-bit) ==")
mdl, tok = carregar(TEACHER_ID, quantizar=True)
ppl_teacher = perplexidade(mdl, tok, teste)
bench_teacher = avaliar_benchmark(mdl, tok, benchmark)
resultados["teacher"] = {"ppl": ppl_teacher, **resumir(bench_teacher)}
liberar(mdl)
print(resultados["teacher"])

== TEACHER 7B (4-bit) ==


tokenizer_config.json: 0.00B [00:00, ?B/s]

vocab.json: 0.00B [00:00, ?B/s]

merges.txt: 0.00B [00:00, ?B/s]

tokenizer.json: 0.00B [00:00, ?B/s]

...

Loading checkpoint shards:   0%|          | 0/4 [00:00<?, ?it/s]

/usr/local/lib/python3.12/dist-packages/bitsandbytes/backends/cuda/ops.py:213: FutureWarning: _check_is_size will be removed in a future PyTorch release along with guard_size_oblivious.     Use _check(i >= 0) instead.
  torch._check_is_size(blocksize)


generation_config.json:   0%|          | 0.00/243 [00:00<?, ?B/s]

/usr/local/lib/python3.12/dist-packages/bitsandbytes/backends/cuda/ops.py:468: FutureWarning: _check_is_size will be removed in a future PyTorch release along with guard_size_oblivious.     Use _check(i >= 0) instead.
  torch._check_is_size(blocksize)


{'ppl': 5.847249466812378, 'rougeL_medio': 0.2120366632233302, 'cobertura_media': 0.675, 'pct_portugues': 1.0}


## 9. Resultado consolidado e veredito de transferência
A conclusão da questão sai daqui: comparar **destilado vs. base**.

In [16]:
import pandas as pd

ordem = ["teacher","student_base","student_destilado"]
rotulos = {"teacher":"Teacher 7B","student_base":"Student 0.5B (base)",
           "student_destilado":"Student 0.5B (destilado)"}
linhas = []
for k in ordem:
    if k in resultados:
        r = resultados[k]
        linhas.append({
            "Variante": rotulos[k],
            "PPL": None if r["ppl"] is None else round(r["ppl"],2),
            "ROUGE-L": round(r["rougeL_medio"],3),
            "Cobertura": round(r["cobertura_media"],3),
            "% PT": round(100*r["pct_portugues"],1),
        })
df = pd.DataFrame(linhas)
print(df.to_string(index=False))
df.to_json("q4_metrics.json", orient="records", force_ascii=False, indent=2)

                Variante  PPL  ROUGE-L  Cobertura  % PT
              Teacher 7B 5.85    0.212      0.675 100.0
     Student 0.5B (base) 6.52    0.175      0.559 100.0
Student 0.5B (destilado) 4.52    0.226      0.409 100.0


In [18]:
# Veredito automático de transferência de conhecimento
if "student_base" in resultados and "student_destilado" in resultados:
    b, d = resultados["student_base"], resultados["student_destilado"]
    dppl = 100*(b["ppl"]-d["ppl"])/b["ppl"]
    drouge = d["rougeL_medio"] - b["rougeL_medio"]
    dpt = d["pct_portugues"] - b["pct_portugues"]
    print(f"Redução de PPL:        {dppl:+.1f}%")
    print(f"Ganho ROUGE-L:         {drouge:+.3f}")
    print(f"Ganho % português:     {100*dpt:+.1f} pp")
    houve = dppl > 0 and (drouge > 0 or dpt > 0)
    print("\nVEREDITO:",
          "houve transferência de conhecimento" if houve
          else "evidência fraca/ausente de transferência")

Redução de PPL:        +30.7%
Ganho ROUGE-L:         +0.051
Ganho % português:     +0.0 pp

VEREDITO: houve transferência de conhecimento


## 10. Comparação qualitativa (5 perguntas-âncora)
Mesmo formato da Q3: lado a lado das variantes, para o relatório.

In [19]:
ancoras = benchmark[:5]
tabela = []
for i, b in enumerate(ancoras):
    linha = {"pergunta": b["pergunta"]}
    if "teacher" in resultados:
        pass  # respostas já em bench_teacher, se gerado
    linha["base"] = bench_base[i]["resposta"][:200]
    linha["destilado"] = bench_dest[i]["resposta"][:200]
    tabela.append(linha)

with open("q4_qualitative.json","w",encoding="utf-8") as f:
    json.dump(tabela, f, ensure_ascii=False, indent=2)

for t in tabela:
    print("P:", t["pergunta"])
    print("BASE:     ", t["base"])
    print("DESTILADO:", t["destilado"])
    print("-"*70)

P: Qual é a diferença entre uma projeção paralela e uma perspectiva?
BASE:      A diferença entre uma projeção paralela e uma perspectiva está na forma como as imagens são representadas no espaço 3D.

1. Projeção Paralela:
   - Esta é a forma mais simples de representar um plano 
DESTILADO: A projeção paralela envolve um plano que se deslocará de forma linearmente para o sentido oposto do plano original, enquanto a perspectiva permite movimentar-se horizontalmente ou verticalmente sobre 
----------------------------------------------------------------------
P: Qual a diferença entre análise qualitativa e quantitativa de dados?
BASE:      A análise qualitativa e quantitative são duas maneiras diferentes de processar e interpretar dados, mas ambas têm seus próprios benefícios e desvantagens.

Análise Qualitativa:
1. Interpretação: É mai
DESTILADO: Análise qualitativa é baseada em observações pessoais ou reflexões individuais sobre o que foi feito ou como se passou; enquanto a análise qua